# Try the trained NLA

Round-trips your own text through the autoencoder:

```
your text  ->  base model, block K residual stream @ final token  ->  activation v
v          ->  AV (verbalizer)   ->  natural-language explanation
explanation->  AR (reconstructor)->  v_hat
                                     FVE(v, v_hat)
```

Defaults to the Qwen2.5-7B-Instruct block-20 NLA. Needs ~1 GPU (~25 GB: base+LoRA
bf16 plus the 21-block critic).

Run from the repo root with the main venv:
`.venv/bin/python -m ipykernel ...`, or `jupyter lab` with `.venv` selected.

In [ ]:
# --- config -----------------------------------------------------------------
from pathlib import Path

BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"
CKPT   = Path.home() / "nla/ckpts25"
DATA   = Path.home() / "nla/data25"

AV_ADAPTER = CKPT / "rl_vllm/iter_000400"      # final RL verbalizer (LoRA on BASE_MODEL)
AR_CKPT    = CKPT / "rl_vllm/critic_latest"    # final RL reconstructor
SIDECAR    = DATA / "rl_shuf.full.parquet"     # the extraction contract
BASELINE_PARQUET = DATA / "av_sft_shuf.full.val.parquet"   # held-out, for the FVE baseline

# To inspect the SFT warm-start instead (before RL), use:
#   AV_ADAPTER = CKPT / "av_sft/iter_0003834"
#   AR_CKPT    = CKPT / "merged/ar_hf"
DEVICE = "cuda"
MAX_NEW_TOKENS = 256

In [ ]:
import torch, numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

from nla.config import load_nla_config
from nla.models import NLACriticModel
from nla.schema import (ACTIVATION_COLUMN, extract_explanation, normalize_activation,
                        resolve_target_scale, compute_predict_mean_baselines)
from nla.utils import critic_predict, register_karvonen_hook
from nla.utils.arch_adapters import resolve_decoder_layers, resolve_text_config

tok = AutoTokenizer.from_pretrained(BASE_MODEL)

# The sidecar IS the contract: marker token ids, prompt templates, scales, and
# the extraction layer. Reading it (rather than hardcoding 20 / 3584) is what
# keeps this notebook correct if you point it at a different NLA.
cfg = load_nla_config(str(SIDECAR), tok)
LAYER = cfg.extraction_layer_index
mse_scale_f = resolve_target_scale(cfg.mse_scale, cfg.d_model)

print(f"base            : {BASE_MODEL}")
print(f"extraction layer: {LAYER}   (output of block {LAYER} == HF hidden_states[{LAYER+1}])")
print(f"d_model         : {cfg.d_model}")
print(f"marker          : {cfg.injection_char!r} id={cfg.injection_token_id}")
print(f"mse_scale       : {mse_scale_f:.3f}")

In [ ]:
# --- load the base model (for extraction) + AV verbalizer --------------------
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, dtype=torch.bfloat16, attn_implementation="sdpa").to(DEVICE).eval()

d_model = resolve_text_config(base.config).hidden_size
assert d_model == cfg.d_model, f"model d_model {d_model} != sidecar {cfg.d_model}"

# The AV is a LoRA on this same base. Sharing one set of base weights would be
# nice, but the injection hook must NOT fire during plain extraction, so we keep
# the verbalizer as a separate PeftModel and only arm its hook while generating.
av = PeftModel.from_pretrained(
    AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype=torch.bfloat16,
                                         attn_implementation="sdpa"),
    str(AV_ADAPTER)).to(DEVICE).eval()

# Karvonen norm-matched injection: the vector is written at the marker token's
# residual stream. vectors_ref is the handoff slot the hook reads each forward.
vectors_ref = [None]
register_karvonen_hook(av, vectors_ref, cfg.injection_token_id,
                       cfg.injection_left_neighbor_id, cfg.injection_right_neighbor_id)

ar = NLACriticModel.from_pretrained(str(AR_CKPT), torch_dtype=torch.bfloat16).to(DEVICE).eval()
print("loaded AV + AR")

In [ ]:
# --- FVE baseline -----------------------------------------------------------
# FVE is only meaningful against a predict-the-mean baseline over a DISTRIBUTION
# of activations, so a single sample can't define one. We use the held-out split
# to compute the same baseline the trainers report against, which makes the
# numbers below directly comparable to the training curves.
import pyarrow.parquet as pq

_t = pq.ParquetFile(str(BASELINE_PARQUET)).read(columns=[ACTIVATION_COLUMN])
_a = _t.column(0).combine_chunks().values.to_numpy().reshape(_t.num_rows, -1)
_, FVE_BASELINE = compute_predict_mean_baselines(
    torch.tensor(_a, dtype=torch.float32), mse_scale_f)
print(f"predict-the-mean baseline MSE = {FVE_BASELINE:.4f}  (n={_t.num_rows} held-out rows)")
del _t, _a

In [ ]:
# --- the three stages -------------------------------------------------------
class _Stop(Exception):
    """Abort the forward once the target layer is captured (skips upper blocks
    and the lm_head, whose logits tensor dominates cost here)."""

@torch.no_grad()
def get_activation(text: str) -> torch.Tensor:
    """Residual stream at `LAYER`'s output, at the FINAL token of `text`. [d_model] fp32."""
    cap = {}
    def hook(_m, _i, out):
        cap["h"] = (out[0] if isinstance(out, tuple) else out).detach()
        raise _Stop
    h = resolve_decoder_layers(base)[LAYER].register_forward_hook(hook)
    try:
        enc = tok(text, return_tensors="pt").to(DEVICE)
        try:
            base(**enc, use_cache=False)
        except _Stop:
            pass
    finally:
        h.remove()
    return cap["h"][0, -1].float()

@torch.no_grad()
def verbalize(act: torch.Tensor) -> str:
    """Activation -> explanation, via the AV with the vector injected at the marker."""
    prompt = cfg.actor_prompt_template.format(injection_char=cfg.injection_char)
    text = tok.apply_chat_template([{"role": "user", "content": prompt}],
                                   tokenize=False, add_generation_prompt=True)
    ids = torch.tensor([tok.encode(text, add_special_tokens=False)], device=DEVICE)
    vectors_ref[0] = act.unsqueeze(0).to(DEVICE)   # armed only for this forward
    try:
        gen = av.generate(input_ids=ids, attention_mask=torch.ones_like(ids),
                          max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                          pad_token_id=tok.eos_token_id)
    finally:
        vectors_ref[0] = None                      # disarm, always
    raw = tok.decode(gen[0, ids.shape[1]:], skip_special_tokens=True)
    return extract_explanation(raw) or raw.strip()

@torch.no_grad()
def reconstruct(explanation: str) -> torch.Tensor:
    """Explanation -> reconstructed activation. [d_model] fp32."""
    prompt = cfg.critic_prompt_template.format(explanation=explanation)
    ids = torch.tensor([tok.encode(prompt, add_special_tokens=False)], device=DEVICE)
    return critic_predict(ar, ids, torch.ones_like(ids), mse_scale_f)[0].float()

def score(v: torch.Tensor, v_hat: torch.Tensor) -> dict:
    """Same normalisation the trainers score under, so FVE is comparable."""
    a = normalize_activation(v.unsqueeze(0), mse_scale_f)
    b = normalize_activation(v_hat.unsqueeze(0), mse_scale_f)
    mse = torch.nn.functional.mse_loss(b, a).item()
    cos = torch.nn.functional.cosine_similarity(a, b).item()
    return {"mse": mse, "fve_pct": 100.0 * (1 - mse / FVE_BASELINE), "cosine": cos,
            "norm_true": v.norm().item(), "norm_pred": v_hat.norm().item()}

def run(text: str, verbose: bool = True) -> dict:
    v = get_activation(text)
    expl = verbalize(v)
    v_hat = reconstruct(expl)
    s = score(v, v_hat)
    if verbose:
        print(f"INPUT  ...{text[-160:]!r}\n")
        print(f"EXPLANATION\n{expl}\n")
        print(f"FVE {s['fve_pct']:.1f}%   cosine {s['cosine']:.3f}   "
              f"mse {s['mse']:.4f}   |v| {s['norm_true']:.1f} -> |v_hat| {s['norm_pred']:.1f}")
    return {"explanation": expl, **s}

## Try it

The activation is taken at the **final token**, so the explanation describes what
the model is "thinking about" *at the end of your text* — write a prefix that
stops somewhere interesting rather than a complete sentence.

Activations were extracted at least 50 tokens into a document, so very short
inputs are out of distribution and will score worse.

In [ ]:
TEXT = (
    "The mitochondrion is often called the powerhouse of the cell. It generates "
    "most of the cell's supply of adenosine triphosphate, which is then used as "
    "a source of chemical energy. In addition to supplying cellular energy, "
    "mitochondria are involved in signaling, cellular differentiation, and cell "
    "death, as well as maintaining control of the cell cycle and"
)
_ = run(TEXT)

In [ ]:
# Your turn
_ = run("""Paste a passage here, ending wherever you want the activation read from""")

## Sanity check against held-out data

Runs the same round-trip on real held-out rows. The mean FVE here should land
near the training curve's final held-out number (**~73%** for the RL checkpoint,
**~62%** for the AR SFT warm-start) — if it is far below, something in the
extraction path above disagrees with how the data was generated.

In [ ]:
import pyarrow.parquet as pq

N = 12
val = pq.ParquetFile(str(BASELINE_PARQUET)).read(
    columns=["detokenized_text_truncated", ACTIVATION_COLUMN]).slice(0, N).to_pylist()

rows = []
for r in val:
    out = run(r["detokenized_text_truncated"], verbose=False)
    # Cross-check our extraction against the vector stored in the dataset: if these
    # disagree, the notebook is reading a different layer/position than datagen did.
    v_stored = torch.tensor(r[ACTIVATION_COLUMN], dtype=torch.float32, device=DEVICE)
    v_ours = get_activation(r["detokenized_text_truncated"])
    out["extract_cos"] = torch.nn.functional.cosine_similarity(
        v_stored.unsqueeze(0), v_ours.unsqueeze(0)).item()
    rows.append(out)

fve = np.array([r["fve_pct"] for r in rows])
cos = np.array([r["cosine"] for r in rows])
ext = np.array([r["extract_cos"] for r in rows])
print(f"round-trip FVE : mean {fve.mean():.1f}%   median {np.median(fve):.1f}%   "
      f"range {fve.min():.1f}..{fve.max():.1f}")
print(f"recon cosine   : mean {cos.mean():.3f}")
print(f"extraction cos vs stored vectors: mean {ext.mean():.4f}  (want > 0.99)")
assert ext.mean() > 0.99, (
    "our extraction disagrees with the dataset's stored activations — check LAYER "
    "(layer_index=K is the OUTPUT of block K == hidden_states[K+1]) and that "
    "BASE_MODEL matches the sidecar's extraction.base_model")

In [ ]:
for r in rows[:4]:
    print(f"FVE {r['fve_pct']:5.1f}%  cos {r['cosine']:.3f}  |  {r['explanation'][:150]}")